# Data Preparation

In [1]:
import os
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from datasets import load_dataset

PROJECT_ROOT = '/content/drive/MyDrive/twincar'
STANFORD_CACHE = f'{PROJECT_ROOT}/stanford_cars_cache'
COMPCARS_PATH = f'{PROJECT_ROOT}/compcars_filtered'
DATA_DIR = f'{PROJECT_ROOT}/data'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load Stanford Cars & Extract Class Map

In [ ]:
dataset = load_dataset('naufalso/stanford_cars', cache_dir=STANFORD_CACHE)

train_data = dataset['train']
test_data  = dataset['test']

train_records = [{'hf_idx': i, 'label': ex['label'], 'car_name': ex['car_name']}
                 for i, ex in enumerate(train_data)]
test_records  = [{'hf_idx': i, 'label': ex['label'], 'car_name': ex['car_name']}
                 for i, ex in enumerate(test_data)]

train_df = pd.DataFrame(train_records)
test_df  = pd.DataFrame(test_records)

del dataset, train_data, test_data

print("Train rows: " + str(len(train_df)))
print("Test rows: " + str(len(test_df)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

Train rows: 8103
Test rows: 8000


### Parse Class Names into Make/Model/Year

In [ ]:
MULTI_WORD_MAKES = ['AM General', 'Aston Martin', 'Land Rover']

def parse_car_name(name):
    for mwm in MULTI_WORD_MAKES:
        if name.startswith(mwm):
            rest = name[len(mwm):].strip().split()
            year = rest[-1] if rest[-1].isdigit() else None
            model = ' '.join(rest[:-1]) if year else ' '.join(rest)
            return mwm, model, year

    tokens = name.strip().split()
    year = tokens[-1] if tokens[-1].isdigit() else None
    make = tokens[0]
    model = ' '.join(tokens[1:-1]) if year else ' '.join(tokens[1:])
    return make, model, year

rows = []
label_to_name = {}
for _, row in train_df.iterrows():
    label_to_name[row['label']] = row['car_name']
for _, row in test_df.iterrows():
    label_to_name[row['label']] = row['car_name']

all_classes = [label_to_name[i] for i in sorted(label_to_name.keys())]
for label, name in enumerate(all_classes):
    make, model, year = parse_car_name(name)
    rows.append({'label': label, 'class_name': name, 'make': make, 'model': model, 'year': year})

class_df = pd.DataFrame(rows)

print("Unique makes: " + str(class_df['make'].nunique()))
print("Unique models: " + str(class_df['model'].nunique()))
print("Year range: " + str(class_df['year'].min()) + " - " + str(class_df['year'].max()))

Unique makes: 48
Unique models: 188
Year range: 1991 - 2012


In [ ]:
class_df.head()

,label,class_name,make,model,year
0,0,AM General Hummer SUV 2000,AM General,Hummer SUV,2000
1,1,Acura RL Sedan 2012,Acura,RL Sedan,2012
2,2,Acura TL Sedan 2012,Acura,TL Sedan,2012
3,3,Acura TL Type-S 2008,Acura,TL Type-S,2008
4,4,Acura TSX Sedan 2012,Acura,TSX Sedan,2012


### Build Train/Val/Test Split from Stanford Cars

The StanfordCars dataset comes with a predefined `train` (8,103 images) and `test` (8,000 images) split.
We keep the `test` split as-is and carve a validation set out of the `train` split.

**Split strategy:**
- `test`: StanfordCars test split (8,000 images, untouched)
- `train`: 85% of StanfordCars train split (~6,887 images)
- `val`: 15% of StanfordCars train split (~1,216 images)

Splitting is **stratified by class label** to ensure every class is represented
proportionally in both train and val, which is important given the class imbalance
we observed in the EDA (min 24, max 68 images per class).

`random_state=42` ensures the split is reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label'],
    random_state=SEED
)

train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print("Stanford split:")
print("  Train : " + str(len(train_split)))
print("  Val: " + str(len(val_split)))
print("  Test: " + str(len(test_df)))
print("  Total: " + str(len(train_split) + len(val_split) + len(test_df)))
print("\nClasses in train: " + str(train_split['label'].nunique()))
print("Classes in val: " + str(val_split['label'].nunique()))

Stanford split:
  Train : 6887
  Val: 1216
  Test: 8000
  Total: 16103

Classes in train: 195
Classes in val: 195


In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

train_split['split'] = 'train'
val_split['split'] = 'val'
test_df['split'] = 'test'

metadata = pd.concat([train_split, val_split, test_df], ignore_index=True)
metadata_path = os.path.join(DATA_DIR, 'stanford_metadata.csv')
metadata.to_csv(metadata_path, index=False)

print("Saved metadata to: " + metadata_path)
print("Total rows: " + str(len(metadata)))
print("Split counts:")
print(metadata['split'].value_counts().to_string())

Saved metadata to: /content/drive/MyDrive/twincar/data/stanford_metadata.csv
Total rows: 16103
Split counts:
split
test     8000
train    6887
val      1216


# CompCars Metadata

## Decision: CompCars Used for Exploration Only

During data preparation, we attempted to unify CompCars labels with Stanford Cars'
195 classes in order to use CompCars images as additional training data.

**The problem:** CompCars provides make and model names, but its naming conventions
differ significantly from Stanford Cars (e.g. `"Benz C Class"` vs `"Mercedes-Benz
C-Class Sedan 2012"`, `"BWM 3 Series"` vs `"BMW 3 Series Sedan 2012"`). Fuzzy
string matching produced only ~32% recall, and many of the matched labels were
incorrect (e.g. `"BMW 3 Series convertible"` mapping to `"BMW 1 Series Convertible"`).

**Why we stopped:** Forcing incorrect labels into training data is worse than having
less data - a mislabeled BMW 3 Series image in the BMW 1 Series class actively
hurts model performance. Cleaning this manually for 34,491 images across 796 model
IDs is not a practical use of project time.

**What CompCars contributed:** The EDA in `01_data_exploration.ipynb` confirmed that
CompCars adds valuable viewpoint diversity (rear, side, rear-side angles) that
Stanford Cars lacks. This motivated our augmentation strategy - we simulate these
viewpoints during training using geometric transforms rather than adding noisy data.

**Going forward:** Training is based on Stanford Cars only (16,103 images, 195 classes).
CompCars remains available for future work, for example as an unlabeled pretraining
corpus or if a proper label alignment tool is developed.